In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import scipy as sp
%matplotlib qt
from continuum_solver import ContinuumSolver, TimeDependentSolver, clean_input
%load_ext autoreload
%autoreload 2

torch.backends.cuda.preferred_linalg_library("magma")
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device is {DEVICE}")
# set default datatype of tensors
DTYPE = torch.complex128

ModuleNotFoundError: No module named 'continuum_solvers'

In [10]:
def harmonic_oscillator(x, x_max=5):
    return 4*(1/2)*(x - x_max/2)**2

def mirror_cavity(x, g_0, x_min, x_max, x_steps):
    dx = (x_max + x_min) / x_steps
    a_mirror = 2 * np.cos(dx) + g_0 * np.sin(dx)
    r = (a_mirror - np.sqrt(a_mirror**2 -  4)) / 2
    g_defect = (2 * r - 2 * np.cos(dx)) / np.sin(dx)

    output = torch.tensor([g_0], device=DEVICE, dtype=DTYPE).repeat(x.shape)
    output[x_steps // 2] = g_defect

    return output

def cavity(x, x_min, x_max, x_steps):
    dx = (x_max + x_min) / x_steps
    g_0 = 2 * (2 - np.cos(dx)) / np.sin(dx)
    g_defect = (2 * (2 - np.sqrt(3)) - 2 * np.cos(dx)) / np.sin(dx)

    output = torch.tensor([g_0], device=DEVICE, dtype=DTYPE).repeat(x.shape)
    output[x_steps // 2] = g_defect

    return output

def potential_fourier(x, x_min, x_max, x_steps, f_steps_min):
    V = torch.zeros((x_steps, 2 * f_steps_min + 1), device=DEVICE, dtype=DTYPE)
    V[:, f_steps_min] = (36 / (torch.pi ** 2)) * (torch.cos(2 * torch.pi * x / (x_max - x_min)) + 1)
    V[:, f_steps_min + 1] = 0 * torch.cos(2 * torch.pi * x / (x_max - x_min))
    V[:, f_steps_min - 1] = 0 * torch.cos(2 * torch.pi * x / (x_max - x_min))

    return V

def fourier_harmonic_oscillator(x, x_min, x_max, x_steps, f_steps_min):
    V = torch.zeros((x_steps, 2 * f_steps_min + 1), device=DEVICE, dtype=DTYPE)
    V[:, f_steps_min] = 2 * (x - (x_max - x_min) / 2)**2

    return V

In [11]:
x_max = 6
x_steps = 1024
f_steps_min = 1
epsilon = 2e-10
k_vals = torch.linspace(0, 2*torch.pi, 3, device=DEVICE)
energies = torch.linspace(0, 2, 50, device=DEVICE, dtype=DTYPE)

solver_ho = ContinuumSolver(
    lambda x: harmonic_oscillator(x, x_max=x_max),
    x_min=0,
    x_max=x_max,
    x_steps=x_steps
)
solver_mirror = ContinuumSolver(
    lambda x: cavity(x, 0, x_max, x_steps),
    x_min=0,
    x_max=x_max,
    x_steps=x_steps
)
solver_time_dep = TimeDependentSolver(
    lambda x, f: fourier_harmonic_oscillator(
        x, x_min=0, x_max=x_max, x_steps=x_steps, f_steps_min=f
    ),
    x_min=0.,
    x_max=x_max,
    x_steps=x_steps,
    f_steps_min=0,
    omega=31.1
)
solver_time_dep_f = TimeDependentSolver(
    lambda x, f: potential_fourier(
        x, x_min=0, x_max=x_max, x_steps=x_steps, f_steps_min=f
    ),
    x_min=0.,
    x_max=x_max,
    x_steps=x_steps,
    f_steps_min=1,
    omega=31.1
)

In [12]:
loss_t_dep, dc_overlap = solver_time_dep.loss(energies, 0)
loss_t_dep_f, dc_overlap_f = solver_time_dep_f.loss(energies, 0)
epsilon = 0.001
dc_overlap[dc_overlap < epsilon] = 0
dc_overlap_f[dc_overlap_f < epsilon] = 0
loss_ho = solver_ho.loss(0, energies)

In [13]:
# solver_time_dep.plot_loss(energies, 0, y_min=-2, y_max=2)

In [14]:
# solver_ho.plot_loss(0, energies)

In [18]:
fig, ax = plt.subplots(figsize=(10, 8))

E_cpu = energies.cpu()

# for i in range(0, loss_t_dep.shape[-1]):
#     ax.scatter(
#         E_cpu,
#         loss_t_dep.cpu()[:, i],
#         label=f"Time-Dependent Loss Col {i}",
#         s=25*dc_overlap.cpu()[:, i]
#     )
for i in range(0, loss_t_dep_f.shape[-1]):
    ax.scatter(
        E_cpu,
        loss_t_dep_f.cpu()[:, i],
        label=f"Time-Dependent F Loss Col {i}",
        s=25*dc_overlap_f.cpu()[:, i]
    )
# ax.plot(E_cpu, loss_t_dep.cpu().sum(dim=-1), label="Time-Dependent Loss Sum")
ax.plot(E_cpu, loss_ho.cpu(), label="Old Loss", color="green")

# ax.legend()
ax.grid()
ax.set_title("Losses vs. Energy Plot")

ax.set_xlabel("Energy")
ax.set_ylabel("Loss")

ax.set_ylim(-1e5, 1e5)

plt.show()

In [16]:
# eye = torch.eye(solver_time_dep.f_steps, device=DEVICE, dtype=DTYPE)
# fourier_coeffs = solver_time_dep.V(solver_time_dep.x_vals, solver_time_dep.f_steps_min)[:, solver_time_dep.f_steps_min]
# V_shape = solver_time_dep.x_vals.shape + solver_time_dep.f_vals.shape